[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/matrices_course.ipynb)

# Matrices and determinants full course: practice notebook

One practice question for every teaching chapter of the video, in the same order, and three cells you can play with
yourself: a cell that builds a matrix from a rule, four sliders that move the unit square and the letter F, and a
system of equations solved with the inverse, one step at a time. Work each question out on paper first, then type
your answer into the box and press **check**. It tells you whether you're right. Press **show working** if you'd like
to see the steps.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Type a matrix in square brackets, row by row, with a semicolon between rows: `[2 -1; 0 3]` is
the matrix with rows 2, −1 and 0, 3. A column is `[3; 4; 5]`. Fractions as `3/2`, and you can put a number in front:
`1/2[3 -1; -4 2]`. An order as `3x4` (rows first). For several answers, separate them with commas, in the order the
question asks. You can also put your answer between the quotes in a cell's last line, `practice(1, "")`, and run the
cell: it checks straight away.

**The cheat sheet.** Every result from the course on two pages:
[matrices cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/matrices_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, re, textwrap
from fractions import Fraction as Fr
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, IntSlider, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # chalk, chalk smudge, ghost lines
CORAL = "#e3896b"                           # where (0, 1) lands: the second column
SKY, MINT, GOLD, VIOLET = "#6fb6bb", "#a0c27c", "#e8c56b", "#b49ad6"   # the first column · a tick · the answer · area

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
NAMES = {"sqrt": sp.sqrt, "pi": sp.pi, "inf": sp.oo, "infinity": sp.oo, "oo": sp.oo}
ORDER = re.compile(r"^\s*(\d+)\s*(?:x|X|×|by)\s*(\d+)\s*$")          # 3x4, 3 × 4, 3 by 4: an order, rows first

def prep(s):
    """Typed symbols → Python: − × ÷ ² ³ √ ∞."""
    s = (s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("²", "^2").replace("³", "^3")
          .replace("∞", "inf"))
    return re.sub(r"√\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", s).replace("√", "sqrt")

def number(s): return parse_expr(s, local_dict=NAMES, transformations=RULES)

def read_matrix(s):
    """'[2 -1; 0 3]', '[[2, -1], [0, 3]]' or '1/2[3 -1; -4 2]' → a sympy Matrix."""
    k = s.index("[")
    pre, body = s[:k].strip().rstrip("*").strip(), s[k:].strip()
    if not body.endswith("]"): raise ValueError("a bracket isn't closed")
    if body.startswith("[["): rows = re.findall(r"\[([^\[\]]*)\]", body[1:-1])
    else: rows = body[1:-1].split(";")
    M = [[number(e) for e in re.split(r"[,\s]+", r.strip()) if e] for r in rows if r.strip()]
    if not M or len({len(r) for r in M}) != 1: raise ValueError("every row needs the same number of entries")
    return (number(pre) if pre else 1) * sp.Matrix(M)

def split_top(s):
    """Split at the commas that aren't inside brackets."""
    parts, depth, cur = [], 0, ""
    for ch in s:
        depth += (ch in "[(") - (ch in "])")
        if ch == "," and depth == 0: parts.append(cur); cur = ""
        else: cur += ch
    return [p.strip() for p in parts + [cur] if p.strip()]

def read_one(p):
    o = ORDER.match(p)
    if o: return (int(o.group(1)), int(o.group(2)))                    # an order
    p = prep(p)
    if "[" in p: return read_matrix(p)                                 # a matrix
    return number(p)                                                   # a number

def read(s):
    """Your typed answer (a list, one for each answer)."""
    return [read_one(p) for p in split_top(s)]

def same(got, want, tol=1e-9):
    if isinstance(want, tuple): return got == want                     # an order
    if isinstance(want, sp.MatrixBase):
        if not isinstance(got, sp.MatrixBase): return False
        if got.shape != want.shape and 1 in want.shape and got.shape == want.T.shape: got = got.T   # a row for a column
        return got.shape == want.shape and all(same(g, w, tol) for g, w in zip(got, want))
    if isinstance(got, (tuple, sp.MatrixBase)): return False
    if want in (sp.oo, -sp.oo): return got == want
    try: return abs(complex(sp.N(sp.sympify(got) - want, 30))) < tol
    except Exception: return False                                     # a letter, not a number

def num(v):
    """A number as you'd write it: 3, −2, 3/2, 0.5."""
    v = sp.nsimplify(v)
    return str(v).replace("-", "−").replace("sqrt", "√").replace("**", "^")

def fmt(M):
    """A matrix as the video prints it: [3/2 −2; −1 5/2]."""
    return "[" + "; ".join(" ".join(num(v) for v in M.row(i)) for i in range(M.rows)) + "]"

def order(M): return f"{M.rows} × {M.cols}"

# Each question: the question, the answers in order, the working, and (if it has several parts) the move that cracks
# each part, shown when that part is wrong.
Q = {}

def practice(n, answer=""):
    """Question n: an answer box, a check button and a show-working button."""
    q = Q[n]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="460px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets, the semicolons and the signs."); return
            want = q["answer"]
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            ok = [same(g, w) for g, w in zip(got, want)]
            if all(ok): print("✓ right"); return
            if "moves" not in q: print("✗ not quite: try again, or press show working"); return
            for k, (good, move) in enumerate(zip(ok, q["moves"]), 1):
                print(f"✓ part {k}: right" if good else f"✗ part {k}: not quite. The move that cracks it: {move}")
    def on_work(_=None):
        out.clear_output()
        with out:
            for line in q["working"]: print(textwrap.fill(line, 110))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"Question {n}. {q['text']}", 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

# Pictures of matrices: a board is an axes measured in inches (1 unit = 72 points), so text and brackets line up.
def board(fig, rect):
    ax = fig.add_axes(rect); ax.axis("off")
    W, H = fig.get_size_inches()
    ax.set_xlim(0, rect[2] * W); ax.set_ylim(0, rect[3] * H)
    return ax

def say(ax, s, x, y, fs=17, col=CHALK, **kw):
    """Text with its left edge at x, centred on y; returns roughly where it ends."""
    ax.text(x, y, s, fontsize=fs, color=col, va="center", ha="left", **kw)
    return x + len(s) * 0.58 * fs / 72

def draw_mat(ax, M, x, y, fs=18, col=CHALK, cols=None, bars=False, hi=None, entry=None):
    """A matrix in square brackets (or determinant bars), its left edge at x, centred on y. cols: a colour for each
    column; hi: {(i, j): colour} for single entries. Returns its right edge."""
    rows = [[(entry or num)(v) for v in M.row(i)] for i in range(M.rows)]
    em = fs / 72
    cw = [max(len(r[j]) for r in rows) * 0.6 * em + 0.75 * em for j in range(M.cols)]
    lh = 1.5 * em; h = M.rows * lh; top = y + h / 2
    xx = x + 0.3 * em
    for j in range(M.cols):
        for i in range(M.rows):
            c = (hi or {}).get((i, j), (cols[j] if cols else col))
            ax.text(xx + cw[j] / 2, top - lh * (i + 0.5), rows[i][j], fontsize=fs, color=c, ha="center", va="center")
        xx += cw[j]
    right = xx + 0.3 * em
    lip = 0 if bars else 0.28 * em
    for bx, s in ((x, 1), (right, -1)):
        ax.plot([bx + s * lip, bx, bx, bx + s * lip], [top + 0.1 * em, top + 0.1 * em, top - h - 0.1 * em, top - h - 0.1 * em],
                color=col, lw=2, solid_capstyle="butt")
    return right

## 1 · Matrices

A matrix is a rectangular array of numbers in rows and columns, written inside square brackets. Its order is m × n:
m rows and n columns, rows always first. The entry aᵢⱼ sits in row i and column j. In the café, the orders matrix
has a row for each friend (Ana, Ben, Kai) and a column for each item (coffee, cake, sandwich), so a₁₁ = 2 is Ana's
coffees.

The next cell builds a matrix from a rule, the way the video builds aᵢⱼ = i + 2j. Change the rule in the box and
press Enter: the grid fills in again. Under each entry you can see the rule with the numbers put in. The two sliders
set the order: m rows and n columns.

In [ ]:
ROW, COL = sp.symbols("i j", integer=True, positive=True)
SUB = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
def sub(k): return str(k).translate(SUB)

def read_rule(rule):
    s = re.sub(r"(\d)\s*([ij])", r"\1*\2", prep(rule))                 # 2j is a number to Python: make it 2*j
    e = parse_expr(s, local_dict={"i": ROW, "j": COL, "sqrt": sp.sqrt, "abs": sp.Abs, "max": sp.Max,
                                           "min": sp.Min}, transformations=RULES)
    if not e.free_symbols <= {ROW, COL}: raise ValueError("only i and j, please")
    return e

def rule_matrix(rule, m, n):
    e = read_rule(rule)
    return sp.Matrix(m, n, lambda r, c: sp.nsimplify(e.subs({ROW: r + 1, COL: c + 1})))

def nice(rule):
    """A rule as you'd write it: 'i^2 - j' → 'i² − j'."""
    s = prep(rule).replace("**", "^").replace("*", "·")
    return s.replace("^2", "²").replace("^3", "³").replace("-", "−")

def shown(rule, i, j):
    """The rule with the numbers put in: 'i + 2j' at row 1, column 3 → '1 + 2·3'."""
    s = prep(rule).replace("**", "^").replace("*", "·")
    s = re.sub(r"([ij])(?=[ij])", r"\1·", s)
    s = re.sub(r"(\d)\s*([ij])\b", r"\1·\2", s)
    s = re.sub(r"\bi\b", str(i), s); s = re.sub(r"\bj\b", str(j), s)
    return s.replace("^2", "²").replace("^3", "³").replace("-", "−")

assert rule_matrix("i + 2j", 2, 3) == sp.Matrix([[3, 5, 7], [4, 6, 8]])     # the video's example
assert shown("i + 2j", 1, 3) == "1 + 2·3" and shown("i^2 - j", 3, 1) == "3² − 1"
assert [(r, 12 // r) for r in range(1, 13) if 12 % r == 0] == [(1, 12), (2, 6), (3, 4), (4, 3), (6, 2), (12, 1)]

def build(rule="i + 2j", m=2, n=3):
    try: A = rule_matrix(rule, m, n)
    except Exception:
        print("I can't read that rule yet. Use i for the row and j for the column, like  i + 2j  or  i^2 - j."); return
    vals = [[num(A[r, c]) for c in range(n)] for r in range(m)]
    work = [[shown(rule, r + 1, c + 1) for c in range(n)] for r in range(m)]
    pitch = max(1.15, max(len(w) for row in work for w in row) * 0.1 + 0.4, max(len(v) for row in vals for v in row) * 0.2 + 0.4)
    W, H = 11.5, 1.45 + 0.95 * m
    fig = plt.figure(figsize=(W, H)); bd = board(fig, [0, 0, 1, 1])
    end = say(bd, f"aᵢⱼ = {nice(rule)}", 0.35, H - 0.45, 20)
    say(bd, f"order {m} × {n}: {m} row{'s' * (m > 1)} and {n} column{'s' * (n > 1)}", max(end + 0.6, 4.6), H - 0.45, 15, DIM)
    x0, top = 0.45, H - 0.95
    for r in range(m):
        for c in range(n):
            last = (r, c) == (m - 1, n - 1)
            cx, cy = x0 + 0.2 + pitch * (c + 0.5), top - 0.95 * (r + 0.48)
            bd.text(cx, cy + 0.1, vals[r][c], fontsize=23, color=GOLD if last else CHALK, ha="center", va="center",
                    fontweight="bold" if last else "normal")
            bd.text(cx, cy - 0.3, work[r][c], fontsize=11, color=GOLD if last else DIM, ha="center", va="center")
    right = x0 + 0.4 + pitch * n
    for bx, s in ((x0, 1), (right, -1)):
        bd.plot([bx + s * 0.15, bx, bx, bx + s * 0.15], [top, top, top - 0.95 * m, top - 0.95 * m], color=CHALK, lw=2.2)
    tx = right + 0.5
    say(bd, f"a{sub(m)}{sub(n)} = {work[-1][-1]} = {vals[-1][-1]}", tx, top - 0.3, 17, GOLD)
    say(bd, f"row {m}, column {n}", tx, top - 0.75, 14, DIM)
    say(bd, f"{m} × {n} = {m * n} entries", tx, top - 1.1, 14, DIM)
    plt.show()

In [ ]:
# Build a matrix from a rule: type a rule for aᵢⱼ (i is the row, j the column), press Enter, and set the order.
W_ = {"style": {"description_width": "initial"}}
interact(build, rule=Text(value="i + 2j", description="rule: aᵢⱼ =", continuous_update=False, **W_),
         m=IntSlider(value=2, min=1, max=5, description="rows m", **W_),
         n=IntSlider(value=3, min=1, max=5, description="columns n", **W_));

In [ ]:
A1 = rule_matrix("2i - j", 3, 4)
Q[1] = {"text": "The matrix A = [aᵢⱼ] has order 3 × 4, and aᵢⱼ = 2i − j. Give the entry a₃₂. Then: five friends at a "
                "café in Lisbon order from a menu of three items. Their orders matrix has one row for each friend and "
                "one column for each item. Give its order (type it like 2x3).",
        "answer": [A1[2, 1], (5, 3)],
        "moves": ["row 3, column 2: put i = 3 and j = 2 into 2i − j.",
                  "rows come first: one row for each friend, then one column for each item."],
        "working": ["a₃₂ is in row 3 and column 2, so put i = 3 and j = 2 into the rule: 2 × 3 − 2 = 4.",
                    f"(The whole matrix is {fmt(A1)}. Try the rule 2i − j with 3 rows and 4 columns in the cell above.)",
                    "Five friends make five rows, and three items make three columns, so the order is 5 × 3.",
                    "It holds 5 × 3 = 15 numbers."]}
assert Q[1]["answer"] == [4, (5, 3)] and A1 == sp.Matrix([[1, 0, -1, -2], [3, 2, 1, 0], [5, 4, 3, 2]])
practice(1)

## 2 · Types of matrices

A row matrix has one row, and a column matrix has one column. A square matrix has as many rows as columns, and its
main diagonal runs from the top left to the bottom right. A diagonal matrix has zeros everywhere off that diagonal.
The identity matrix I has ones on the diagonal and zeros elsewhere, and the zero matrix O is all zeros. Two matrices
are equal when they have the same order and every pair of matching entries is equal.

In [ ]:
x, y = sp.symbols("x y")
EQ2 = sp.solve([sp.Eq(x + y, 9), sp.Eq(x - 2 * y, 0)], [x, y])
Q[2] = {"text": "[x + y 3; 7 x − 2y] = [9 3; 7 0]. Find x and y. Then: how many entries of the 4 × 4 identity matrix "
                "are zero?",
        "answer": [EQ2[x], EQ2[y], list(sp.eye(4)).count(0)],
        "moves": ["match the top left entries: x + y = 9.",
                  "match the bottom right entries: x − 2y = 0, so x = 2y.",
                  "16 entries, and only the main diagonal holds ones."],
        "working": ["Equal matrices: every pair of matching entries is equal.",
                    "Top left: x + y = 9. Bottom right: x − 2y = 0, so x = 2y.",
                    "Put x = 2y into the first: 3y = 9, so y = 3 and x = 6.",
                    "Check: 6 + 3 = 9 and 6 − 6 = 0. (The 3s and the 7s already match.)",
                    "The 4 × 4 identity has 16 entries. The four on the main diagonal are ones, so 16 − 4 = 12 are zero."]}
assert Q[2]["answer"] == [6, 3, 12]
practice(2)

## 3 · Addition and scalar multiples

Add or subtract two matrices of the same order place by place: (A + B)ᵢⱼ = aᵢⱼ + bᵢⱼ. Matrices of different orders
can't be added. To multiply a matrix by a number k, multiply every entry by k. Matrix equations solve like ordinary
ones: from 2X + A = B, X = ½(B − A).

In [ ]:
C3 = sp.Matrix([[2, 1, 0], [0, 1, 1], [1, 0, 1]])          # the café: rows Ana, Ben, Kai; columns coffee, cake, sandwich
W3 = sp.Matrix([[1, 0, 1], [2, 1, 0], [0, 2, 1]])          # the weekend's extra orders
A3, B3 = sp.Matrix([[1, 3], [2, 0]]), sp.Matrix([[6, -1], [4, 7]])
X3 = (B3 - A3) / 2
Q[3] = {"text": "One day's café orders are C = [2 1 0; 0 1 1; 1 0 1] (rows Ana, Ben, Kai; columns coffee, cake, "
                "sandwich). They order the same for 5 days, and then the weekend adds W = [1 0 1; 2 1 0; 0 2 1]. Find "
                "5C + W. Then find the matrix X with 2X + A = B, where A = [1 3; 2 0] and B = [6 −1; 4 7].",
        "answer": [5 * C3 + W3, X3],
        "moves": ["multiply every entry of C by 5, then add W place by place.",
                  "take A from both sides, then halve: X = ½(B − A)."],
        "working": [f"5C = {fmt(5 * C3)}: every entry times 5.",
                    f"5C + W = {fmt(5 * C3 + W3)}, place by place: Ana's coffees are 10 + 1 = 11.",
                    f"2X + A = B, so 2X = B − A = {fmt(B3 - A3)}.",
                    f"Halve every entry: X = {fmt(X3)}.",
                    f"Check: 2X + A = {fmt(2 * X3)} + {fmt(A3)} = {fmt(2 * X3 + A3)} = B."]}
assert Q[3]["answer"][0] == sp.Matrix([[11, 5, 1], [2, 6, 5], [5, 2, 6]]) and 2 * X3 + A3 == B3
assert X3 == sp.Matrix([[sp.Rational(5, 2), -2], [1, sp.Rational(7, 2)]])
practice(3)

## 4 · Matrix multiplication

To multiply, go along a row of the first matrix and down a column of the second, multiply the pairs and add:
(AB)ᵢⱼ = row i of A · column j of B. That's exactly how the café bills came out: Ana's row 2, 1, 0 times the prices
3, 4, 5 is 6 + 4 + 0 = 10. The inside sizes must match: (m × n)(n × p) = m × p. And the order matters: AB and BA are
usually different.

In [ ]:
CAFE, PRICES = sp.Matrix([[2, 1, 0], [0, 1, 1], [1, 0, 1]]), sp.Matrix([3, 4, 5])
assert list(CAFE * PRICES) == [10, 9, 8]                        # the video's bills
O4 = sp.Matrix([[1, 2, 1], [3, 0, 2]])                          # Lea, Omar: coffee, cake, sandwich
P4 = sp.Matrix([[3, 2], [4, 5], [5, 4]])                        # prices at café 1 and café 2
A4, B4 = sp.Matrix([[1, 2], [0, 1]]), sp.Matrix([[3, 0], [1, 1]])
Q[4] = {"text": "Lea orders 1 coffee, 2 cakes and 1 sandwich. Omar orders 3 coffees and 2 sandwiches. At café 1 a "
                "coffee, a cake and a sandwich cost 3, 4 and 5; at café 2 they cost 2, 5 and 4. With the orders as "
                "O = [1 2 1; 3 0 2] and the prices as P = [3 2; 4 5; 5 4], find OP (what each friend pays at each "
                "café). Then, with A = [1 2; 0 1] and B = [3 0; 1 1], find AB − BA.",
        "answer": [O4 * P4, A4 * B4 - B4 * A4],
        "moves": ["(2 × 3)(3 × 2) = 2 × 2. Lea at café 1: 1 × 3 + 2 × 4 + 1 × 5.",
                  "work out AB and BA separately, each entry a row times a column, then subtract."],
        "working": ["OP is (2 × 3)(3 × 2) = 2 × 2: a row for each friend, a column for each café.",
                    "Lea: café 1, 3 + 8 + 5 = 16; café 2, 2 + 10 + 4 = 16. The same at both!",
                    "Omar: café 1, 9 + 0 + 10 = 19; café 2, 6 + 0 + 8 = 14.",
                    f"So OP = {fmt(O4 * P4)}.",
                    f"AB = {fmt(A4 * B4)} and BA = {fmt(B4 * A4)}: not the same.",
                    f"AB − BA = {fmt(A4 * B4 - B4 * A4)}."]}
assert Q[4]["answer"] == [sp.Matrix([[16, 16], [19, 14]]), sp.Matrix([[2, -4], [0, -2]])]
practice(4)

## 5 · Matrices as transformations

A matrix moves points: write a point as a column and multiply. The columns of a 2 × 2 matrix [a b; c d] are where the
two unit steps land: (1, 0) goes to (a, c), the first column, and (0, 1) goes to (b, d), the second. So the unit
square becomes a parallelogram, and any shape is carried along with it. The grid paper stays still. To build the
matrix for a move, decide where the two steps should go.

Move the four sliders: a and b are the top row, c and d the bottom row. The sky arrow is the first column and the
coral arrow the second. The parallelogram's area is the determinant ad − bc (its size, with a minus sign when the F is
flipped over). Try building a quarter turn, then a reflection.

In [ ]:
F_SHAPE = np.array([(0.25, 0.1), (0.4, 0.1), (0.4, 0.45), (0.62, 0.45), (0.62, 0.58), (0.4, 0.58), (0.4, 0.75),
                    (0.75, 0.75), (0.75, 0.9), (0.25, 0.9)])         # a letter F inside the unit square
LIM = 3.3                                                            # the plane runs from −3.3 to 3.3 (big moves run off it)
MOVES = {((1, 0), (0, 1)): "the identity: nothing moves",
         ((0, -1), (1, 0)): "a quarter turn anticlockwise", ((0, 1), (-1, 0)): "a quarter turn clockwise",
         ((-1, 0), (0, -1)): "a half turn", ((1, 0), (0, -1)): "a reflection in the x axis",
         ((-1, 0), (0, 1)): "a reflection in the y axis", ((0, 1), (1, 0)): "a reflection in the line y = x",
         ((0, -1), (-1, 0)): "a reflection in the line y = −x"}

def g(v):
    """A slider value as you'd write it: 1, 0.5, −1.5."""
    return f"{v:g}".replace("-", "−")

def gb(v): return f"({g(v)})" if v < 0 else g(v)                     # bracketed when negative, for products

def move_name(a, b, c, d):
    key = ((a, b), (c, d))
    if key in MOVES: return MOVES[key]
    if a == b == c == d == 0: return "everything lands on the origin"
    if b == c == 0 and a == d: return f"an enlargement, scale factor {g(a)}"
    if b == c == 0 and a * d != 0: return f"a stretch: × {g(a)} across, × {g(d)} up"
    if a == d == 1 and c == 0: return f"a shear: the top edge slides {g(b)}"
    if a == d == 1 and b == 0: return f"a shear: the right edge slides {g(c)} up"
    if a * d - b * c == 0: return "squashed flat onto a line"
    return ""

def arrow(ax, p, col, lw=3.2):
    if abs(p[0]) + abs(p[1]) < 1e-9: return
    ax.annotate("", xy=p, xytext=(0, 0), zorder=6,
                arrowprops=dict(arrowstyle="-|>", color=col, lw=lw, mutation_scale=22, shrinkA=0, shrinkB=0))

for (a_, b_), (c_, d_) in MOVES:                                      # each named move, checked:
    M_ = sp.Matrix([[a_, b_], [c_, d_]])
    assert M_ * M_.T == sp.eye(2) and abs(M_.det()) == 1                # turns and mirrors keep lengths and areas
assert sp.Matrix([[0, -1], [1, 0]]).det() == 1 and sp.Matrix([[0, 1], [1, 0]]).det() == -1   # a turn keeps the F's way round
assert list(sp.Matrix([[3, 1], [0, 2]]) * sp.Matrix([1, 1])) == [4, 2] and sp.Matrix([[3, 1], [0, 2]]).det() == 6   # the video's M

def transform(a=1.0, b=0.0, c=0.0, d=1.0):
    Mx = np.array([[a, b], [c, d]]); D = a * d - b * c
    fig = plt.figure(figsize=(11.5, 5.0))
    ax = fig.add_axes([0.035, 0.06, 0.43, 0.91])
    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect("equal")
    ax.set_xticks(range(-3, 4)); ax.set_yticks(range(-3, 4)); ax.tick_params(labelsize=10, length=0)
    for s_ in ax.spines.values(): s_.set_visible(False)
    ax.axhline(0, color=DIM, lw=1.3); ax.axvline(0, color=DIM, lw=1.3)
    sq = np.array([(0, 0), (1, 0), (1, 1), (0, 1)])
    ax.fill(*sq.T, fc="none", ec=DIM, ls="--", lw=1.3)                    # where the square started
    ax.fill(*F_SHAPE.T, fc="none", ec=DIM, lw=1.0, alpha=0.6)             # … and the F
    img, Fi = sq @ Mx.T, F_SHAPE @ Mx.T
    ax.fill(*img.T, fc=VIOLET, alpha=0.32, ec=VIOLET, lw=2.2, zorder=3)
    ax.fill(*Fi.T, fc=CHALK, alpha=0.92, ec=CHALK, lw=1.4, zorder=4)
    arrow(ax, (a, c), SKY); arrow(ax, (b, d), CORAL)
    ax.text(-LIM + 0.12, LIM - 0.15, f"violet area = {g(abs(D))}", color=GOLD, fontsize=13, fontweight="bold",
            ha="left", va="top", zorder=8, bbox=dict(boxstyle="round,pad=0.3", fc=SLATE, ec="none", alpha=0.85))
    bd = board(fig, [0.48, 0.0, 0.52, 1.0])
    say(bd, "the matrix", 0.1, 4.6, 14, DIM)
    right = draw_mat(bd, sp.Matrix([[a, b], [c, d]]), 0.1, 3.8, 24, cols=[SKY, CORAL], entry=lambda v: g(float(v)))
    say(bd, f"(1, 0) lands on ({g(a)}, {g(c)}): the first column", right + 0.35, 4.05, 13.5, SKY)
    say(bd, f"(0, 1) lands on ({g(b)}, {g(d)}): the second column", right + 0.35, 3.55, 13.5, CORAL)
    say(bd, f"det = ad − bc = {gb(a)}·{gb(d)} − {gb(b)}·{gb(c)} = {g(D)}", 0.1, 2.7, 18, GOLD, fontweight="bold")
    say(bd, f"the area of the square: 1 → {g(abs(D))}", 0.1, 2.15, 15)
    say(bd, ("positive: the F still reads the right way round" if D > 0 else
             "negative: the F is flipped over" if D < 0 else
             "zero: the square is squashed flat, area 0"), 0.1, 1.7, 15, CHALK if D else CORAL)
    name = move_name(a, b, c, d)
    if name: say(bd, name, 0.1, 0.95, 19, CORAL, fontweight="bold")
    plt.show()

SL = dict(min=-2, max=2, step=0.5, readout_format=".1f", style={"description_width": "initial"})
MA_, MB_, MC_, MD_ = (FloatSlider(value=v, description=k, **SL) for k, v in (("a", 1), ("b", 0), ("c", 0), ("d", 1)))

In [ ]:
interact(transform, a=MA_, b=MB_, c=MC_, d=MD_);

In [ ]:
TURN_CW = sp.Matrix([[0, 1], [-1, 0]])                             # (1, 0) → (0, −1), (0, 1) → (1, 0)
REF_Y, ENL2 = sp.Matrix([[-1, 0], [0, 1]]), sp.Matrix([[2, 0], [0, 2]])
Q[5] = {"text": "Give the matrix of a quarter turn clockwise about the origin. Then give the single matrix that "
                "reflects in the y axis first, and then enlarges by scale factor 2.",
        "answer": [TURN_CW, ENL2 * REF_Y],
        "moves": ["where do (1, 0) and (0, 1) land after a quarter turn clockwise? Those points are the columns.",
                  "B first, then A, is the product AB: the enlargement on the left, the reflection on the right."],
        "working": ["A quarter turn clockwise sends (1, 0) to (0, −1), and (0, 1) to (1, 0).",
                    f"Those are the two columns, so the matrix is {fmt(TURN_CW)}. (Try it with the sliders.)",
                    f"Reflection in the y axis: (1, 0) → (−1, 0) and (0, 1) stays, so R = {fmt(REF_Y)}.",
                    f"Enlargement by 2: E = {fmt(ENL2)}.",
                    f"Reflect first, then enlarge: E R = {fmt(ENL2 * REF_Y)}."]}
assert list(TURN_CW * sp.Matrix([1, 0])) == [0, -1] and list(TURN_CW * sp.Matrix([0, 1])) == [1, 0]
assert Q[5]["answer"][1] == sp.Matrix([[-2, 0], [0, 2]])
practice(5)

## 6 · Transpose and symmetric matrices

The transpose Aᵀ turns rows into columns: (Aᵀ)ᵢⱼ = aⱼᵢ. Transposing twice gives A back, and the transpose of a product
reverses the order: (AB)ᵀ = BᵀAᵀ. A matrix is symmetric when Aᵀ = A, and skew-symmetric when Aᵀ = −A (its diagonal is
then all zeros). Every square matrix splits into a symmetric part and a skew-symmetric part:
A = ½(A + Aᵀ) + ½(A − Aᵀ).

In [ ]:
A6 = sp.Matrix([[3, 8], [2, 5]])
SYM6, SKW6 = (A6 + A6.T) / 2, (A6 - A6.T) / 2
M6 = sp.Matrix([[1, x], [4, 2]])
X6 = sp.solve(M6[0, 1] - M6[1, 0], x)[0]                             # symmetric: the top right equals the bottom left
Q[6] = {"text": "A = [3 8; 2 5]. Find its symmetric part ½(A + Aᵀ) and its skew-symmetric part ½(A − Aᵀ). Then: for "
                "which x is [1 x; 4 2] symmetric?",
        "answer": [SYM6, SKW6, X6],
        "moves": ["Aᵀ = [3 2; 8 5]; add it to A, then halve.",
                  "take Aᵀ from A, then halve: the diagonal comes out zero.",
                  "symmetric means the matrix equals its transpose: the entry above the diagonal matches the one below."],
        "working": [f"Aᵀ = {fmt(A6.T)}: the rows of A become columns.",
                    f"A + Aᵀ = {fmt(A6 + A6.T)}, so the symmetric part is {fmt(SYM6)}.",
                    f"A − Aᵀ = {fmt(A6 - A6.T)}, so the skew-symmetric part is {fmt(SKW6)}.",
                    f"Check: {fmt(SYM6)} + {fmt(SKW6)} = {fmt(SYM6 + SKW6)} = A.",
                    "[1 x; 4 2] is symmetric when its transpose [1 4; x 2] is the same matrix, so x = 4."]}
assert SYM6 + SKW6 == A6 and SYM6.T == SYM6 and SKW6.T == -SKW6 and X6 == 4 and M6.subs(x, X6).is_symmetric()
assert SYM6 == sp.Matrix([[3, 5], [5, 5]]) and SKW6 == sp.Matrix([[0, 3], [-3, 0]])
practice(6)

## 7 · Determinants

The determinant of a 2 × 2 matrix is det A = |a b; c d| = ad − bc. It's how much the matrix scales area: the unit
square becomes a parallelogram of area |ad − bc|, and every shape's area is multiplied by the same number. A negative
determinant means the shape is flipped over. A zero determinant means the plane is squashed onto a line. (The bars
mean determinant, not size: a determinant can be negative.)

In [ ]:
k = sp.Symbol("k")
D7, T7 = sp.Matrix([[5, 2], [3, 4]]), sp.Matrix([[2, 1], [1, 3]])
K7 = sp.solve(sp.Matrix([[k, 6], [2, 3]]).det(), k)
Q[7] = {"text": "Find |5 2; 3 4|. Then: the matrix [2 1; 1 3] moves a triangle of area 4. What is the area of the "
                "triangle it lands on? Then find k so that [k 6; 2 3] has determinant zero.",
        "answer": [D7.det(), abs(T7.det()) * 4, K7[0]],
        "moves": ["ad − bc: 5 × 4 − 2 × 3.",
                  "every area is multiplied by the determinant: find |2 1; 1 3| first.",
                  "set ad − bc = 0: 3k − 12 = 0."],
        "working": ["|5 2; 3 4| = 5 × 4 − 2 × 3 = 20 − 6 = 14.",
                    "|2 1; 1 3| = 6 − 1 = 5, so every area is multiplied by 5.",
                    "The triangle of area 4 lands on a triangle of area 4 × 5 = 20.",
                    "|k 6; 2 3| = 3k − 12. That's zero when k = 4.",
                    "(At k = 4 the columns (4, 2) and (6, 3) point the same way, so the square is squashed flat.)"]}
assert Q[7]["answer"] == [14, 20, 4] and len(K7) == 1
practice(7)

## 8 · Determinant of a 3×3 matrix

Expand along a row: each entry times the 2 × 2 determinant left when you cover its row and column, with the signs
+ − + along the top row. The signs follow a chessboard pattern, so you can expand along any row or column and get the
same answer. Choose the one with the most zeros. A 3 × 3 determinant is how much the matrix scales volume.

In [ ]:
E8a = sp.Matrix([[2, 0, 1], [3, 1, 0], [1, 4, 2]])
E8b = sp.Matrix([[1, 0, 2], [3, 0, 5], [2, 7, 1]])
Q[8] = {"text": "Find |2 0 1; 3 1 0; 1 4 2|. Then find |1 0 2; 3 0 5; 2 7 1| (choose the column with two zeros).",
        "answer": [E8a.det(), E8b.det()],
        "moves": ["along the top row: 2 × |1 0; 4 2| − 0 × (…) + 1 × |3 1; 1 4|.",
                  "down column 2 only the 7 counts. Its sign: row 3, column 2 is a minus place."],
        "working": ["Along the top row, signs + − +:",
                    "2 × |1 0; 4 2| − 0 × |3 0; 1 2| + 1 × |3 1; 1 4| = 2 × 2 − 0 + 1 × (12 − 1) = 4 + 11 = 15.",
                    "Down column 2: the two zeros give nothing, so only the 7 counts.",
                    "Row 3, column 2 has the sign (−1)³⁺² = −1. Cover its row and column: |1 2; 3 5| = 5 − 6 = −1.",
                    "So the determinant is −7 × (−1) = 7."]}
assert Q[8]["answer"] == [15, 7] and sp.Matrix([[1, 2], [3, 5]]).det() == -1
assert E8b.det() == -7 * E8b.minor(2, 1) and E8a.det() == 2 * 2 + 1 * 11                  # down column 2; the top row
practice(8)

## 9 · Properties of determinants

|Aᵀ| = |A|. Swapping two rows changes the sign. Two equal (or proportional) rows give zero. Multiplying one row by k
multiplies the determinant by k, so for an n × n matrix |kA| = kⁿ|A|. Adding a multiple of one row to another changes
nothing. A triangular matrix's determinant is the product of its diagonal. And |AB| = |A||B|.

In [ ]:
dA = 3
P9 = sp.Matrix([[1, 2, 3], [2, 4, 6], [7, 8, 9]])
Q[9] = {"text": "A is a 3 × 3 matrix with |A| = 3. Find |2A| and |A²|. Then find |1 2 3; 2 4 6; 7 8 9| without "
                "expanding.",
        "answer": [2 ** 3 * dA, dA * dA, P9.det()],
        "moves": ["2A doubles every row, and there are three rows: 2³ × |A|, not 2 × |A|.",
                  "A² = A × A, and |AB| = |A||B|.",
                  "look at rows 1 and 2."],
        "working": ["2A multiplies each of the three rows by 2, so |2A| = 2³ × 3 = 8 × 3 = 24.",
                    "|A²| = |A| × |A| = 3 × 3 = 9.",
                    "Row 2 is twice row 1. Take the 2 out, and two rows are equal, so the determinant is 0."]}
assert Q[9]["answer"] == [24, 9, 0] and P9.row(1) == 2 * P9.row(0)
for _ in range(3):                                                     # the rules, checked on random matrices
    R_ = sp.randMatrix(3, 3, -5, 5)
    assert (2 * R_).det() == 8 * R_.det() and (R_ * R_).det() == R_.det() ** 2 and R_.T.det() == R_.det()
practice(9)

## 10 · Area of a triangle

The triangle with corners (x₁, y₁), (x₂, y₂), (x₃, y₃) has area ½ |x₁ y₁ 1; x₂ y₂ 1; x₃ y₃ 1|, taking the size (the
determinant is negative when the corners go clockwise). It's half the parallelogram on two of its sides. Three points
lie on one line exactly when this determinant is zero.

In [ ]:
def tri(P1, P2, P3): return sp.Matrix([[*P1, 1], [*P2, 1], [*P3, 1]]).det()
CAFES = ((1, 2), (5, 3), (2, 7))
K10 = sp.solve(tri((0, 1), (2, 5), (4, k)), k)
Q[10] = {"text": "On a city map (in kilometres), three cafés stand at (1, 2), (5, 3) and (2, 7). Find the area of the "
                 "triangle between them. Then a delivery bike rides in a straight line through (0, 1) and (2, 5). Find "
                 "k so that it also passes (4, k).",
         "answer": [abs(tri(*CAFES)) / 2, K10[0]],
         "moves": ["½ |1 2 1; 5 3 1; 2 7 1|: expand along the top row.",
                   "three points on one line: |0 1 1; 2 5 1; 4 k 1| = 0."],
         "working": ["|1 2 1; 5 3 1; 2 7 1| = 1 × (3 − 7) − 2 × (5 − 2) + 1 × (35 − 6) = −4 − 6 + 29 = 19.",
                     "The area is ½ × 19 = 19/2 = 9.5 square kilometres.",
                     "|0 1 1; 2 5 1; 4 k 1| = 0 × (5 − k) − 1 × (2 − 4) + 1 × (2k − 20) = 2k − 18.",
                     "That's zero when k = 9. (Check: the slope from (0, 1) to (2, 5) is 2, and 1 + 2 × 4 = 9.)"]}
assert Q[10]["answer"] == [sp.Rational(19, 2), 9] and tri(*CAFES) == 19
assert tri((1, 1), (4, 2), (2, 5)) == 11                               # the video's triangle: area 5.5
practice(10)

## 11 · Minors and cofactors

The minor Mᵢⱼ is the determinant left when you cover row i and column j. The cofactor is Cᵢⱼ = (−1)ⁱ⁺ʲ Mᵢⱼ: the minor
with the chessboard sign. A determinant is any row's entries times their own cofactors, added up. A row times another
row's cofactors gives zero.

In [ ]:
A11 = sp.Matrix([[2, 1, 3], [0, 4, 1], [5, 2, 6]])
Q[11] = {"text": "A = [2 1 3; 0 4 1; 5 2 6]. Find the minor M₂₃, the cofactor C₂₃ and the cofactor C₃₁.",
         "answer": [A11.minor(1, 2), A11.cofactor(1, 2), A11.cofactor(2, 0)],
         "moves": ["cover row 2 and column 3: |2 1; 5 2|.",
                   "C₂₃ = (−1)²⁺³ M₂₃: a minus place.",
                   "cover row 3 and column 1: |1 3; 4 1|. Row 3, column 1 is a plus place."],
         "working": ["Cover row 2 and column 3: M₂₃ = |2 1; 5 2| = 4 − 5 = −1.",
                     "C₂₃ = (−1)⁵ × (−1) = 1.",
                     "Cover row 3 and column 1: M₃₁ = |1 3; 4 1| = 1 − 12 = −11.",
                     "C₃₁ = (−1)⁴ × (−11) = −11.",
                     f"(Check with row 2: 0 × C₂₁ + 4 × C₂₂ + 1 × C₂₃ = {A11.cofactor(1, 1) * 4} + 1 = {A11.det()} = |A|.)"]}
assert Q[11]["answer"] == [-1, 1, -11] and 4 * A11.cofactor(1, 1) + 1 * A11.cofactor(1, 2) == A11.det()
practice(11)

## 12 · Adjoint and inverse

The inverse undoes a matrix: A A⁻¹ = A⁻¹ A = I. For a 2 × 2, [a b; c d]⁻¹ = 1/(ad − bc) [d −b; −c a]: swap the
diagonal, change the signs of the other two, and divide by the determinant. In general the adjoint, adj A, is the
transpose of the matrix of cofactors, A · adj A = |A| I, and so A⁻¹ = adj A / |A|. There's an inverse only when
|A| ≠ 0. And (AB)⁻¹ = B⁻¹A⁻¹, |adj A| = |A|ⁿ⁻¹.

In [ ]:
K12, L12 = sp.Matrix([[4, 7], [1, 2]]), sp.Matrix([[2, 1], [4, 3]])
Q[12] = {"text": "A secret-message key is K = [4 7; 1 2]. Find K⁻¹. Then find the inverse of [2 1; 4 3]. Then: a "
                 "3 × 3 matrix A has |A| = 5. Find |adj A|.",
         "answer": [K12.inv(), L12.inv(), 5 ** (3 - 1)],
         "moves": ["|K| = 8 − 7 = 1: swap the diagonal, change the signs of the other two.",
                   "the determinant is 6 − 4 = 2: swap, change signs, then divide every entry by 2.",
                   "|adj A| = |A|ⁿ⁻¹ with n = 3."],
         "working": ["|K| = 4 × 2 − 7 × 1 = 1, so K⁻¹ = [2 −7; −1 4] (swap the 4 and 2, change the signs of 7 and 1).",
                     f"Check: K K⁻¹ = {fmt(K12 * K12.inv())} = I.",
                     "|2 1; 4 3| = 6 − 4 = 2, so the inverse is 1/2 [3 −1; −4 2] = [3/2 −1/2; −2 1].",
                     "|adj A| = |A|ⁿ⁻¹ = 5² = 25 for a 3 × 3."]}
assert Q[12]["answer"][:2] == [sp.Matrix([[2, -7], [-1, 4]]), sp.Matrix([[sp.Rational(3, 2), -sp.Rational(1, 2)], [-2, 1]])]
assert K12 * K12.inv() == sp.eye(2) and Q[12]["answer"][2] == 25
for A_ in (sp.diag(1, 1, 5), sp.Matrix([[1, 2, 0], [0, 1, 1], [2, 0, 1]])):              # any 3 × 3 with |A| = 5
    assert A_.det() == 5 and A_.adjugate().det() == 25
practice(12)

## 13 · Systems of linear equations

A system of equations is one matrix equation, A X = B: A holds the coefficients, X the unknowns, B the constants.
When |A| ≠ 0, multiply both sides by A⁻¹ on the left: X = A⁻¹B. The café's lost prices come back this way, from the
bills 10, 9 and 8.

The next cell solves a system with the inverse. Move the step slider to watch each step: the set-up, the
determinant, the adjoint, the inverse, the multiplication, and the check. Then change A and B to a system of your
own, and run the cell again.

In [ ]:
STEPS = ["write it as A X = B", "find |A|", "find adj A", "A⁻¹ = adj A ÷ |A|", "X = A⁻¹ B", "check: A X = B"]

def expansion(A):
    """|A| worked out the way the video does it: ad − bc, or along the top row with signs + − +."""
    if A.rows == 2: return f"{num(A[0, 0])}·{num(A[1, 1])} − {num(A[0, 1])}·{num(A[1, 0])}"
    if A.rows == 3:
        t = [f"{num(A[0, j])}·({num(A.minor(0, j))})" for j in range(3)]
        return f"{t[0]} − {t[1]} + {t[2]}"
    return "…"

def solve_steps(step=1):
    n = A.rows
    assert A.is_square and B.shape == (n, 1), "A must be square, and B a column as tall as A"
    X = sp.Matrix(sp.symbols("x y z") if n == 3 else sp.symbols("x y") if n == 2 else sp.symbols(f"x1:{n + 1}"))
    D = A.det()
    fig = plt.figure(figsize=(11.5, 4.4)); bd = board(fig, [0, 0, 1, 1])
    for s, name in enumerate(STEPS, 1):
        col = CORAL if s == step else CHALK if s < step else GHOST
        say(bd, f"{s}  {name}", 0.25, 3.9 - 0.6 * (s - 1), 14, col, fontweight="bold" if s == step else "normal")
    bd.plot([3.35, 3.35], [0.5, 4.1], color=GHOST, lw=1.2)
    x0, y0, fs = 3.65, 2.4, 20
    if step == 1:
        r = draw_mat(bd, A, x0, y0, fs); r = draw_mat(bd, X, r + 0.15, y0, fs, GOLD)
        r = say(bd, "=", r + 0.2, y0, 20); draw_mat(bd, B, r + 0.15, y0, fs)
        say(bd, "A holds the coefficients, X the unknowns (gold), B the constants", x0, 0.65, 13, DIM)
    elif D == 0:
        say(bd, "|A| = 0", x0, 3.2, 22, CORAL, fontweight="bold")
        say(bd, "A squashes the plane flat, so it has no inverse,", x0, 2.4, 15)
        say(bd, "and this method can't finish. Chapter 15 says what happens:", x0, 1.95, 15)
        say(bd, "no solution, or infinitely many.", x0, 1.5, 15)
    elif step == 2:
        r = draw_mat(bd, A, x0, y0 + 0.2, fs, bars=True)
        say(bd, f"= {expansion(A)} = {num(D)}", r + 0.2, y0 + 0.2, 17)
        say(bd, f"|A| = {num(D)}: not zero, so A has an inverse", x0, 0.75, 16, GOLD, fontweight="bold")
    elif step == 3:
        if n == 2:
            say(bd, "swap the diagonal, change the signs of the other two:", x0, 3.6, 14, DIM)
            draw_mat(bd, A.adjugate(), x0 + 1.1, y0, fs, GOLD); say(bd, "adj A =", x0, y0, 17)
        else:
            say(bd, "the cofactors, then rows become columns:", x0, 3.85, 14, DIM)
            r = draw_mat(bd, A.cofactor_matrix(), x0, y0, fs)
            r = say(bd, "→ transpose →", r + 0.2, y0, 14, DIM)
            draw_mat(bd, A.adjugate(), r + 0.15, y0, fs, GOLD)
            say(bd, "adj A (gold): the transpose of the matrix of cofactors", x0, 0.65, 13, DIM)
    elif step == 4:
        r = say(bd, f"A⁻¹ = 1/{num(D)}" if D != 1 else "A⁻¹ =", x0, y0, 17)
        r = draw_mat(bd, A.adjugate(), r + 0.1, y0, fs); r = say(bd, "=", r + 0.2, y0, 20)
        draw_mat(bd, A.inv(), r + 0.15, y0, fs, GOLD)
        say(bd, "divide every entry of adj A by |A|", x0, 0.65, 13, DIM)
    elif step == 5:
        Xv = A.inv() * B
        r = say(bd, "X =", x0, y0, 17); r = draw_mat(bd, A.inv(), r + 0.1, y0, fs)
        r = draw_mat(bd, B, r + 0.12, y0, fs); r = say(bd, "=", r + 0.2, y0, 20); draw_mat(bd, Xv, r + 0.15, y0, fs, GOLD)
        say(bd, ",  ".join(f"{v} = {num(w)}" for v, w in zip(X, Xv)), x0, 0.65, 18, GOLD, fontweight="bold")
    else:
        Xv = A.inv() * B
        r = draw_mat(bd, A, x0, y0, fs); r = draw_mat(bd, Xv, r + 0.12, y0, fs, GOLD)
        r = say(bd, "=", r + 0.2, y0, 20); r = draw_mat(bd, A * Xv, r + 0.15, y0, fs)
        ok = A * Xv == B
        say(bd, "✓ that's B: every equation checks" if ok else "✗ that isn't B", x0, 0.65, 17, MINT if ok else CORAL,
            fontweight="bold")
    plt.show()

# the video's numbers: the café system, |A| = 3, adj A, and the prices 3, 4, 5
_A, _B = sp.Matrix([[2, 1, 0], [0, 1, 1], [1, 0, 1]]), sp.Matrix([10, 9, 8])
assert _A.det() == 3 and _A.adjugate() == sp.Matrix([[1, -1, 1], [1, 2, -2], [-1, 1, 2]])
assert list(_A.adjugate() * _B) == [9, 12, 15] and list(_A.inv() * _B) == [3, 4, 5] and expansion(_A) == "2·(1) − 1·(−1) + 0·(−1)"
assert list(sp.Matrix([[2, 1], [5, 3]]).inv() * sp.Matrix([7, 18])) == [3, 1]          # the video's small system

In [ ]:
# A system as A X = B: the café's orders and bills (the prices are lost). Change A and B, and run the cell again.
A = sp.Matrix([[2, 1, 0],
               [0, 1, 1],
               [1, 0, 1]])
B = sp.Matrix([10, 9, 8])
interact(solve_steps, step=IntSlider(value=1, min=1, max=6, description="step"));

In [ ]:
A13 = sp.Matrix([[1, 1, 2], [2, 1, 0], [0, 3, 1]])
B13 = sp.Matrix([15, 13, 18])
X13 = A13.inv() * B13
Q[13] = {"text": "At a juice bar in Mexico City, Mia buys 1 juice, 1 wrap and 2 cookies for 15. Leo buys 2 juices and 1 "
                 "wrap for 13. Sam buys 3 wraps and 1 cookie for 18. With x, y and z the prices of a juice, a wrap and "
                 "a cookie: x + y + 2z = 15, 2x + y = 13, 3y + z = 18. Solve for x, y and z (put A and B in the cell "
                 "above to watch it, after you've tried it on paper).",
         "answer": list(X13),
         "moves": ["A = [1 1 2; 2 1 0; 0 3 1], |A| = 11. Then X = A⁻¹B = adj A · B ÷ 11."] * 3,
         "working": ["A = [1 1 2; 2 1 0; 0 3 1], X = [x; y; z], B = [15; 13; 18].",
                     "|A| = 1 × (1 − 0) − 1 × (2 − 0) + 2 × (6 − 0) = 1 − 2 + 12 = 11.",
                     f"adj A = {fmt(A13.adjugate())}.",
                     f"adj A · B = {fmt(A13.adjugate() * B13)}, and ÷ 11 gives X = {fmt(X13)}.",
                     "So a juice costs 4, a wrap 5 and a cookie 3.",
                     "Check: 4 + 5 + 6 = 15, 8 + 5 = 13, 15 + 3 = 18."]}
assert A13.det() == 11 and list(X13) == [4, 5, 3] and A13 * X13 == B13
practice(13)

## 14 · Cramer's rule

Cramer's rule solves one unknown at a time. Let D be the determinant of the coefficient matrix. Replace its first
column with the constants to get Dx, its second column to get Dy (and its third to get Dz). Then x = Dx/D, y = Dy/D,
z = Dz/D. It works when D ≠ 0, and each unknown is a ratio of two areas (or volumes).

In [ ]:
A14, B14 = sp.Matrix([[2, 3], [1, -1]]), sp.Matrix([13, -1])
def swap_col(A, j, b): M = A.copy(); M[:, j] = b; return M
D14, DX14, DY14 = A14.det(), swap_col(A14, 0, B14).det(), swap_col(A14, 1, B14).det()
Q[14] = {"text": "At a café in Nairobi, 2 coffees and 3 muffins cost 13, and a coffee costs 1 less than a muffin: "
                 "2x + 3y = 13 and x − y = −1. Use Cramer's rule: give D, Dx and Dy, then x and y.",
         "answer": [D14, DX14, DY14, DX14 / D14, DY14 / D14],
         "moves": ["D = |2 3; 1 −1|.", "Dx: the constants 13, −1 in place of the first column.",
                   "Dy: the constants in place of the second column.", "x = Dx/D.", "y = Dy/D."],
         "working": ["D = |2 3; 1 −1| = 2 × (−1) − 3 × 1 = −5.",
                     "Dx = |13 3; −1 −1| = −13 + 3 = −10.",
                     "Dy = |2 13; 1 −1| = −2 − 13 = −15.",
                     "x = −10/−5 = 2 and y = −15/−5 = 3: a coffee costs 2 and a muffin 3.",
                     "Check: 4 + 9 = 13 and 2 − 3 = −1."]}
assert Q[14]["answer"] == [-5, -10, -15, 2, 3] and A14 * sp.Matrix([2, 3]) == B14
A3c, B3c = sp.Matrix([[1, 1, 1], [2, -1, 1], [1, 2, -1]]), sp.Matrix([6, 3, 2])          # the video's 3 × 3 example
assert A3c.det() == 7 and [swap_col(A3c, j, B3c).det() for j in range(3)] == [7, 14, 21]
practice(14)

## 15 · Consistency of a system

A system is consistent when it has at least one solution, and inconsistent when it has none. It never has exactly
two. The test: if |A| ≠ 0 there's exactly one solution, X = A⁻¹B. If |A| = 0, work out (adj A)B. If that isn't zero,
there's no solution. If it is zero, there may be infinitely many solutions or none, so check the equations themselves.

In [ ]:
A15 = lambda k: sp.Matrix([[1, 3], [2, k]])
B15 = sp.Matrix([5, 7])
K15 = sp.solve(A15(k).det(), k)[0]
AB15 = A15(K15).adjugate() * B15
Q[15] = {"text": "Two equations: x + 3y = 5 and 2x + ky = 7. Find the value of k that makes |A| = 0. For that k, "
                 "work out (adj A)B (a column). How many solutions does the system have then? (Type 0, 1 or inf.)",
         "answer": [K15, AB15, 0],
         "moves": ["|1 3; 2 k| = k − 6.",
                   "adj [1 3; 2 6] = [6 −3; −2 1]; multiply it by the column [5; 7].",
                   "|A| = 0 and (adj A)B isn't zero."],
         "working": ["|A| = |1 3; 2 k| = k − 6, which is zero when k = 6.",
                     "With k = 6: adj A = [6 −3; −2 1] (swap the diagonal, change the signs of the other two).",
                     "(adj A)B = [6 × 5 − 3 × 7; −2 × 5 + 1 × 7] = [9; −3]. Not zero.",
                     "So the system has no solution: it's inconsistent.",
                     "You can see why: 2x + 6y = 7 says x + 3y = 3.5, but the first equation says x + 3y = 5.",
                     "The two lines are parallel."]}
assert K15 == 6 and AB15 == sp.Matrix([9, -3]) and sp.linsolve((A15(6), B15)) == sp.EmptySet
assert sp.Matrix([[1, 2], [2, 4]]).adjugate() * sp.Matrix([4, 10]) == sp.Matrix([-4, 2])     # the video's example
practice(15)

## Sources
NCERT Mathematics Class XII, ch. 3 (Matrices) and ch. 4 (Determinants); OpenStax, College Algebra 2e, 7.5 (Matrices
and Matrix Operations), 7.7 (Solving Systems with Inverses) and 7.8 (Solving Systems with Cramer's Rule); Pearson
Edexcel A level Further Mathematics (9FM0), Core Pure, Matrices. All retrieved 2026-10-31. Every question is our own.